In [ ]:
# SRC_DIR/
# ├── trn/
# │   ├── l-<loc_id>/             DST_DIR/
# │   │   ├── <rel_id>.jpg        ├── test_images/test_images/
# │   │   ...                     │   ├── <fname>.jpg
# │   ...                    ->   |   ...
# └── tst/                        └── yolo_dataset/yolo_dataset/train/images/
#     ├── l-<loc_id>/                 ├── <fname>.jpg
#     │   ├── <rel_id>.jpg            ...
#     │   ...
#     ...

In [ ]:
import os
import shutil
from pathlib import Path
import pandas as pd

MTRN = pd.read_csv('./trn_meta.csv', parse_dates=['datetime'])
MTST = pd.read_csv('./tst_meta.csv', parse_dates=['datetime'])

SRC_DIR = Path('./attn-sewn')
DST_DIR = Path('./sewn-dataset')

In [ ]:
test_dest = DST_DIR / 'test_images' / 'test_images'
train_dest = DST_DIR / 'yolo_dataset' / 'yolo_dataset' / 'train' / 'images'

test_dest.mkdir(parents=True, exist_ok=True)
train_dest.mkdir(parents=True, exist_ok=True)


def process_split(split_name, df, target_dir):
    split_dir = SRC_DIR / split_name
    if not split_dir.exists(): raise RuntimeError("Нет папки")
    for loc_dir in split_dir.iterdir():
        if not loc_dir.is_dir() or not loc_dir.name.startswith('l-'):
            continue
        loc_val = loc_dir.name
        loc_val = int(loc_val.split('-')[1])
        sub_df = df[df['loc'] == loc_val]
        fnames = sub_df['fname'].tolist()
        for img_file in loc_dir.glob('*.jpg'):
            img_id = int(img_file.stem)
            if img_id >= len(fnames): raise RuntimeError("айдишник не попал")
            new_fname = fnames[img_id]
            if not new_fname.lower().endswith('.jpg'):
                new_fname += '.jpg'
            src_path = img_file
            dst_path = target_dir / new_fname
            shutil.copy2(src_path, dst_path)

In [ ]:
process_split('trn', MTRN, train_dest)
process_split('tst', MTST, test_dest)